In [37]:
import numpy as np
import pandas as pd

cafe_sales = pd.read_csv('dirty_cafe_sales.csv', na_values=['UNKNOWN', 'ERROR', ' '])
cafe_sales['Transaction Date'] = pd.to_datetime(cafe_sales['Transaction Date'])
cafe_sales = cafe_sales.rename(columns={
    'Transaction ID': 'transaction_id',
    'Item': 'item',
    'Quantity': 'quantity',
    'Price Per Unit': 'price_per_unit',
    'Total Spent': 'total_spent',
    'Payment Method': 'payment_method',
    'Location': 'location',
    'Transaction Date': 'transaction_date'
})

print(cafe_sales.isna().sum())


transaction_id         0
item                 969
quantity             479
price_per_unit       533
total_spent          502
payment_method      3178
location            3961
transaction_date     460
dtype: int64


In [38]:
cafe_sales['price_per_unit'].unique()

array([2. , 3. , 1. , 5. , 4. , 1.5, nan])

In [39]:
cafe_sales['item'].dropna().unique()

price = cafe_sales['price_per_unit'].dropna().sort_values().unique()

for i in price:
    items = cafe_sales.loc[cafe_sales['price_per_unit'] == i, 'item'].unique()
    print(i, items)

1.0 <StringArray>
['Cookie', nan]
Length: 2, dtype: str
1.5 <StringArray>
[nan, 'Tea']
Length: 2, dtype: str
2.0 <StringArray>
['Coffee', nan]
Length: 2, dtype: str
3.0 <StringArray>
['Cake', nan, 'Juice']
Length: 3, dtype: str
4.0 <StringArray>
['Smoothie', 'Sandwich', nan]
Length: 3, dtype: str
5.0 <StringArray>
['Salad', nan]
Length: 2, dtype: str


In [40]:
cafe_sales['total_spent'] = cafe_sales['total_spent'].fillna(cafe_sales['quantity']*cafe_sales['price_per_unit'])
cafe_sales['quantity'] = cafe_sales['quantity'].fillna(cafe_sales['total_spent']/cafe_sales['price_per_unit'])
cafe_sales['price_per_unit'] = cafe_sales['price_per_unit'].fillna(cafe_sales['total_spent']/cafe_sales['quantity'])


In [41]:
def filling_na_price (df, x, y):
    df.loc[df['price_per_unit'] == x, 'item'] = df.loc[df['price_per_unit'] == x, 'item'].fillna(y)

items_prices = {
    'Cookie' : 1.0,
    'Tea' : 1.5,
    'Coffee' : 2.0,
    'Salad' : 5.0
}

for item, price in items_prices.items():
    filling_na_price(cafe_sales, price, item)


In [42]:
all_items = {
    1.0 : 'Cookie',
    1.5 : 'Tea',
    2.0 : 'Coffee',
    3.0 : ['Cake', 'Juice'],
    4.0 : ['Smoothie', 'Sandwich'],
    5.0 : 'Salad'
}

def filling_items (df, x, y):
    condition = (
        df['item'].isin(x)
        if isinstance(x, list)
        else df['item'] == x
    )

    df.loc[condition, 'price_per_unit'] = (
        df.loc[condition, 'price_per_unit']
        .fillna(y)
    )

for price, item in all_items.items():
    filling_items(cafe_sales, item, price)



In [43]:
cafe_sales['total_spent'] = cafe_sales['total_spent'].fillna(cafe_sales['quantity']*cafe_sales['price_per_unit'])
cafe_sales['quantity'] = cafe_sales['quantity'].fillna(cafe_sales['total_spent']/cafe_sales['price_per_unit'])
cafe_sales['price_per_unit'] = cafe_sales['price_per_unit'].fillna(cafe_sales['total_spent']/cafe_sales['quantity'])


In [44]:
mask = cafe_sales['price_per_unit'].isin([3.0, 4.0]), 'item'
cafe_sales.loc[mask] = cafe_sales.loc[mask].fillna('Ambiguous')

In [45]:
def outlier_check (x):
    Q1 = np.nanpercentile(x, 25)
    Q3 = np.nanpercentile(x, 75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return lower_bound, upper_bound



numeric_cols = [
    cafe_sales['quantity'],
    cafe_sales['price_per_unit'],
    cafe_sales['total_spent']
]

for i in numeric_cols:
    print(outlier_check(i))
    lower, upper = outlier_check(i)
    
    print(cafe_sales[(i < lower) | (i > upper)])

(np.float64(-1.0), np.float64(7.0))
Empty DataFrame
Columns: [transaction_id, item, quantity, price_per_unit, total_spent, payment_method, location, transaction_date]
Index: []
(np.float64(-1.0), np.float64(7.0))
Empty DataFrame
Columns: [transaction_id, item, quantity, price_per_unit, total_spent, payment_method, location, transaction_date]
Index: []
(np.float64(-8.0), np.float64(24.0))
     transaction_id   item  quantity  price_per_unit  total_spent  \
10      TXN_2548360  Salad       5.0             5.0         25.0   
51      TXN_6342161  Salad       5.0             5.0         25.0   
52      TXN_8914892  Salad       5.0             5.0         25.0   
96      TXN_5220895  Salad       5.0             5.0         25.0   
100     TXN_9517146  Salad       5.0             5.0         25.0   
...             ...    ...       ...             ...          ...   
9791    TXN_1232346  Salad       5.0             5.0         25.0   
9805    TXN_9506076  Salad       5.0             5.0     

In [46]:
for i in numeric_cols:
    print(cafe_sales[i <= 0])

Empty DataFrame
Columns: [transaction_id, item, quantity, price_per_unit, total_spent, payment_method, location, transaction_date]
Index: []
Empty DataFrame
Columns: [transaction_id, item, quantity, price_per_unit, total_spent, payment_method, location, transaction_date]
Index: []
Empty DataFrame
Columns: [transaction_id, item, quantity, price_per_unit, total_spent, payment_method, location, transaction_date]
Index: []


In [47]:
cafe_sales[(i < lower) | (i > upper)].head(50)

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
51,TXN_6342161,Salad,5.0,5.0,25.0,NaN,Takeaway,2023-01-08
52,TXN_8914892,Salad,5.0,5.0,25.0,Digital Wallet,NaN,2023-03-15
96,TXN_5220895,Salad,5.0,5.0,25.0,Cash,In-store,2023-06-10
100,TXN_9517146,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-10-30
150,TXN_8687151,Salad,5.0,5.0,25.0,Cash,NaN,2023-06-10
157,TXN_4283157,Salad,5.0,5.0,25.0,Digital Wallet,In-store,2023-11-25
177,TXN_1896955,Salad,5.0,5.0,25.0,NaN,In-store,2023-09-23
214,TXN_8693704,Salad,5.0,5.0,25.0,Cash,In-store,2023-05-04
330,TXN_5523450,Salad,5.0,5.0,25.0,Credit Card,Takeaway,2023-07-02


In [48]:
filled_mask = (cafe_sales['total_spent'].notna()) & (cafe_sales['price_per_unit'].notna()) & (cafe_sales['quantity'].notna())

cafe_sales.loc[filled_mask, 'total_spent'] == cafe_sales.loc[filled_mask, 'quantity'] * cafe_sales.loc[filled_mask, 'price_per_unit']
cafe_sales.loc[filled_mask, 'quantity'] == cafe_sales.loc[filled_mask, 'total_spent'] / cafe_sales.loc[filled_mask, 'price_per_unit']
cafe_sales.loc[filled_mask, 'price_per_unit'] == cafe_sales.loc[filled_mask, 'total_spent'] / cafe_sales.loc[filled_mask, 'quantity']

0       True
1       True
2       True
3       True
4       True
        ... 
9995    True
9996    True
9997    True
9998    True
9999    True
Length: 9974, dtype: bool

In [49]:
print(cafe_sales.isna().sum())

transaction_id         0
item                   6
quantity              23
price_per_unit         6
total_spent           23
payment_method      3178
location            3961
transaction_date     460
dtype: int64


In [50]:
cafe_sales.to_csv('cleaned_cafe_sales_dataset.csv', index=False)